# Capstone Two: Pre-processing and Training Data Development
## Class A/B/C Multifamily Risk Score in USA
**Author:** Daksha Gummadi

This notebook prepares my data for machine learning modeling by creating dummy variables for categorical features, standardizing numeric features and splitting the data into training and testing sets.

# Table of Contents

1. [Import Libraries](#1-import-libraries)
2. [Load Data](#2-load-data)
3. [Examine Data Structure](#3-examine-data-structure)
4. [Create Dummy Variables](#4-create-dummy-variables)
5. [Standardize Numeric Features](#5-standardize-numeric-features)
6. [Split into Training and Testing Sets](#6-split-into-training-and-testing-sets)
7. [Save Preprocessed Data](#7-save-preprocessed-data)
8. [Summary](#8-summary)

## 1. Import Libraries

I import the libraries needed for pre-processing:
- pandas: for data manipulation
- numpy: for numerical operations
- sklearn.preprocessing: for scaling numeric features
- sklearn.model_selection: for splitting data into train and test sets

In [28]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

## 2. Load Data

I load the cleaned dataset from my data wrangling step. This dataset contains 72 months of multifamily rental data with rent levels, growth rates and property class information.

In [30]:
df = pd.read_csv('master_dataset_cleaned.csv')
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (72, 14)


,date,avg_rent,vacancy_rate,permits,rent_growth_yoy,rent_class_a,rent_class_b,rent_class_c,growth_class_a,growth_class_b,growth_class_c,year,month,quarter
0,2020-01-31,1144.559911,6.600000,1495.0,NaN,1652.213252,1076.384644,773.257105,NaN,NaN,NaN,2020,1,1
1,2020-02-29,1150.840542,6.300000,1455.0,NaN,1660.939300,1083.146596,776.129677,NaN,NaN,NaN,2020,2,1
2,2020-03-31,1157.791469,6.000000,1346.0,NaN,1666.230657,1091.209584,782.516050,NaN,NaN,NaN,2020,3,1
3,2020-04-30,1155.929327,5.700000,1076.0,NaN,1653.767831,1090.998773,784.205937,NaN,NaN,NaN,2020,4,2
4,2020-05-31,1154.101083,5.933333,1250.0,NaN,1643.310542,1091.453494,786.572517,NaN,NaN,NaN,2020,5,2


## 3. Examine Data Structure

Before pre-processing I need to understand what types of variables I have. I check for categorical variables that need dummy encoding and numeric variables that need scaling.

In [32]:
# Check data types
print("Data types:")
print(df.dtypes)
print("\nDataset info:")
df.info()

Data types:
date                object
avg_rent           float64
vacancy_rate       float64
permits            float64
rent_growth_yoy    float64
rent_class_a       float64
rent_class_b       float64
rent_class_c       float64
growth_class_a     float64
growth_class_b     float64
growth_class_c     float64
year                 int64
month                int64
quarter              int64
dtype: object

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   date             72 non-null     object 
 1   avg_rent         72 non-null     float64
 2   vacancy_rate     70 non-null     float64
 3   permits          70 non-null     float64
 4   rent_growth_yoy  60 non-null     float64
 5   rent_class_a     72 non-null     float64
 6   rent_class_b     72 non-null     float64
 7   rent_class_c     72 non-null     float64
 8   growth_class_

In [33]:
# Identify categorical and numeric columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"\nNumeric columns ({len(numeric_cols)}): {numeric_cols}")

Categorical columns (1): ['date']

Numeric columns (13): ['avg_rent', 'vacancy_rate', 'permits', 'rent_growth_yoy', 'rent_class_a', 'rent_class_b', 'rent_class_c', 'growth_class_a', 'growth_class_b', 'growth_class_c', 'year', 'month', 'quarter']


In [34]:
# Check ranges of numeric features
print("Numeric feature ranges:")
for col in numeric_cols:
    if df[col].notna().sum() > 0:
        print(f"{col}: {df[col].min():.2f} to {df[col].max():.2f}")

Numeric feature ranges:
avg_rent: 1144.48 to 1370.30
vacancy_rate: 5.60 to 7.20
permits: 1076.00 to 1920.00
rent_growth_yoy: -1.72 to 11.44
rent_class_a: 1590.34 to 1989.50
rent_class_b: 1076.38 to 1292.02
rent_class_c: 773.26 to 915.47
growth_class_a: -1.37 to 14.48
growth_class_b: -1.80 to 11.72
growth_class_c: -2.25 to 6.81
year: 2020.00 to 2025.00
month: 1.00 to 12.00
quarter: 1.00 to 4.00


I have one categorical column which is the date column. However date is not a feature I will use for modeling so I do not need to create dummy variables for it. Instead I will drop it before modeling since I already extracted the useful time information into year, month and quarter columns. My numeric features have very different ranges. Average rent is around 1,144 to 1,370 while growth rates are -1.72 to 11.44. Year ranges from 2020 to 2025. These different magnitudes mean I need to standardize the features so they all have similar scales for machine learning algorithms to work properly.

## 4. Create Dummy Variables

Machine learning algorithms need all features to be numeric. I check if I have any categorical variables that need to be converted to dummy indicator variables. Since my only categorical column is date which I will not use for modeling I will drop it instead of creating dummies.

In [80]:
#check for categorical variables
categorical_cols = df_model.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns found: {categorical_cols}")

if len(categorical_cols) > 0:
    #reates dummy variables
    df_dummies = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)
    print(f"\nShape after creating dummies: {df_dummies.shape}")
    
else:
    print("\nNo categorical variables found that need dummy encoding,")
    print("my data is already fully numeric after dropping the date column.")
    df_dummies = df_model

Categorical columns found: []

No categorical variables found that need dummy encoding,
my data is already fully numeric after dropping the date column.


In [37]:
# Drop date column since I already have year, month, quarter
df_model = df.drop(columns=['date'])
print(f"Shape after dropping date: {df_model.shape}")
print(f"\nRemaining columns: {df_model.columns.tolist()}")

Shape after dropping date: (72, 13)

Remaining columns: ['avg_rent', 'vacancy_rate', 'permits', 'rent_growth_yoy', 'rent_class_a', 'rent_class_b', 'rent_class_c', 'growth_class_a', 'growth_class_b', 'growth_class_c', 'year', 'month', 'quarter']


In [38]:
#verifies all columns are now numeric
print("All columns numeric?")
print(df_model.dtypes)

All columns numeric?
avg_rent           float64
vacancy_rate       float64
permits            float64
rent_growth_yoy    float64
rent_class_a       float64
rent_class_b       float64
rent_class_c       float64
growth_class_a     float64
growth_class_b     float64
growth_class_c     float64
year                 int64
month                int64
quarter              int64
dtype: object


I do not have traditional categorical variables like gender, city names or product categories that need dummy encoding. My only object type column was date which is a timestamp. Since I already extracted the useful time information into separate year, month and quarter columns I simply drop the date column. All my remaining features are already numeric and ready for scaling.

## 5. Standardize Numeric Features

My numeric features have very different scales. Rent values are in the thousands while growth rates are percentages between -2 and 14. Machine learning algorithms work better when all features have similar scales so I use StandardScaler to standardize them. This transforms each feature to have a mean of 0 and standard deviation of 1.

In [41]:
print("Missing values before scaling:")
print(df_model.isnull().sum())

Missing values before scaling:
avg_rent            0
vacancy_rate        2
permits             2
rent_growth_yoy    12
rent_class_a        0
rent_class_b        0
rent_class_c        0
growth_class_a     12
growth_class_b     12
growth_class_c     12
year                0
month               0
quarter             0
dtype: int64


In [42]:
#drop rows with missing values for modeling
#the first 12 rows have NaN in growth columns which is expected
# vacancy and permits have 2 NaN at the end
df_clean = df_model.dropna()
print(f"Shape after dropping NaN: {df_clean.shape}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")

Shape after dropping NaN: (58, 13)
Missing values: 0


In [73]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_clean)
df_scaled = pd.DataFrame(scaled_data, columns=df_clean.columns)
df_scaled.head()

,avg_rent,vacancy_rate,permits,rent_growth_yoy,rent_class_a,rent_class_b,rent_class_c,growth_class_a,growth_class_b,growth_class_c,year,month,quarter
0,-2.905934,0.745140,1.855216,-0.454552,-2.932781,-3.045362,-2.249234,-0.891147,-0.173011,0.453493,-1.391331,-1.576018,-1.315071
1,-2.872046,0.346816,0.919259,-0.563530,-2.883590,-2.994841,-2.274460,-0.946792,-0.288064,0.146534,-1.391331,-1.280196,-1.315071
2,-2.749981,-0.051507,1.211746,-0.582395,-2.763534,-2.863724,-2.181066,-0.885398,-0.345190,-0.152752,-1.391331,-0.984374,-1.315071
3,-2.473592,-0.449831,0.984256,-0.209909,-2.515378,-2.520365,-2.008959,-0.468271,0.038996,0.137923,-1.391331,-0.688552,-0.407046
4,-2.176008,-0.715380,0.867262,0.188194,-2.217100,-2.132611,-1.996002,-0.017393,0.455215,-0.026433,-1.391331,-0.392729,-0.407046


In [44]:
print("Mean of scaled features (should be close to 0):")
print(df_scaled.mean())
print("\nStandard deviation of scaled features (should be close to 1):")
print(df_scaled.std())

Mean of scaled features (should be close to 0):
avg_rent           5.512832e-16
vacancy_rate      -2.411864e-16
permits           -9.570888e-17
rent_growth_yoy    1.454775e-16
rent_class_a      -4.641881e-15
rent_class_b       1.565797e-15
rent_class_c      -2.557341e-15
growth_class_a    -2.297013e-17
growth_class_b     3.828355e-17
growth_class_c    -1.339924e-16
year               1.127068e-14
month              1.914178e-17
quarter            1.339924e-16
dtype: float64

Standard deviation of scaled features (should be close to 1):
avg_rent           1.008734
vacancy_rate       1.008734
permits            1.008734
rent_growth_yoy    1.008734
rent_class_a       1.008734
rent_class_b       1.008734
rent_class_c       1.008734
growth_class_a     1.008734
growth_class_b     1.008734
growth_class_c     1.008734
year               1.008734
month              1.008734
quarter            1.008734
dtype: float64


**What scaling does:**

Before scaling my features had very different ranges. Average rent was around 1,200 while growth rates were around 2. This means rent values would dominate machine learning algorithms simply because they are larger numbers. After scaling all features are transformed to have mean 0 and standard deviation 1. Now a value of 1.5 in scaled rent means the same thing as 1.5 in scaled growth - both are 1.5 standard deviations above their respective means. This puts all features on equal footing for modeling.

## 6. Split into Training and Testing Sets

I split my data into training and testing sets using an 80/20 split. The training set will be used to fit machine learning models. The testing set will be held out to evaluate how well the models perform on unseen data. I use random_state=42 so the split is reproducible.

In [47]:
# Define features (X) and target (y)
# For now I'll use all columns as features
# Later in modeling I'll select which ones to use as predictors vs target
X = df_scaled

print(f"Features shape: {X.shape}")
print(f"Features: {X.columns.tolist()}")

Features shape: (58, 13)
Features: ['avg_rent', 'vacancy_rate', 'permits', 'rent_growth_yoy', 'rent_class_a', 'rent_class_b', 'rent_class_c', 'growth_class_a', 'growth_class_b', 'growth_class_c', 'year', 'month', 'quarter']


In [48]:
# Split into train (80%) and test (20%)
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")
print(f"\nTrain/test split: {len(X_train)}/{len(X_test)} ({len(X_train)/len(X)*100:.0f}%/{len(X_test)/len(X)*100:.0f}%)")

Training set size: (46, 13)
Testing set size: (12, 13)

Train/test split: 46/12 (79%/21%)


I split into 80% training and 20% testing which is a standard split ratio. The training set has 45 observations and the testing set has 13 observations. I will use the training set to build and tune my risk scoring model. The testing set stays completely separate and untouched until the very end when I want to see how well my final model performs on new data it has never seen before. This tests whether my model generalizes well or if it just memorized the training data.

## 7. Save Preprocessed Data

I save the preprocessed training and testing sets as CSV files so I can load them easily in the modeling step without having to repeat all the pre-processing work.

In [51]:
# Save training and testing sets
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)

print("Saved files:")
print("  - X_train.csv (training set)")
print("  - X_test.csv (testing set)")

Saved files:
  - X_train.csv (training set)
  - X_test.csv (testing set)


In [52]:
# Also save the scaler for future use
import pickle

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Saved scaler object for future use")

Saved scaler object for future use


I save the scaler object because if I ever need to make predictions on new data in the future I need to scale it the exact same way I scaled the training data. The scaler remembers the mean and standard deviation it learned from the training data. This ensures any new data gets transformed consistently with the training data.

## 8. Conclusion

I completed pre processing and training data development for my multifamily risk score capstone project. Here is what I did:

- I didn't need to create dummy variables because my dataset does not have traditional categorical features. My only object type column was date which I dropped since I already extracted year, month and quarter as numeric features.
- I used StandardScaler to standardize all numeric features to have mean 0 and standard deviation 1. This puts features like rent (values around 1,200) and growth rates (values around 2) on the same scale so they contribute equally to machine learning algorithms.
- I split the data into 80% training (45 observations) and 20% testing (13 observations) using random_state=42 for reproducibility. The training set will be used to build models and the testing set will evaluate performance on unseen data.
- I saved the preprocessed training and testing sets as CSV files and saved the scaler object as a pickle file. I can now load these files in the modeling step without repeating the pre-processing work.

My data is now ready for modeling. The next step will be to build risk scoring models using the preprocessed training data.